# ThermoRG-NN Phase A — Full Theory Validation Pipeline
## Training + TAS Metrics + d_manifold + Publication Plots

**断点续传**: 每个架构独立保存，CSV 存在则跳过；finally 块保证结果总是推回 GitHub。

---

In [1]:
import os, sys, subprocess
from kaggle_secrets import UserSecretsClient

## ▶️ Step 1: Environment & Code Clone

In [2]:
%cd /kaggle/working
!rm -rf ThermoRG-NN
user_secrets = UserSecretsClient()
gh_token = user_secrets.get_secret("gh_token")
repo_url = f"https://{gh_token}@github.com/xliu203/ThermoRG-NN.git"
!git clone {repo_url}
%cd /kaggle/working/ThermoRG-NN
!git config --global user.email "xliu203@asu.edu"
!git config --global user.name "Leo Liu"
print("Code cloned and identity configured")

/kaggle/working
Cloning into 'ThermoRG-NN'...
remote: Enumerating objects: 264, done.
remote: Counting objects: 100% (264/264), done.
remote: Compressing objects: 100% (191/191), done.
remote: Total 264 (delta 61), reused 253 (delta 52), pack-reused 0 (from 0)
Receiving objects: 100% (264/264), 294.86 KiB | 3.98 MiB/s, done.
Resolving deltas: 100% (61/61), done.
/kaggle/working/ThermoRG-NN
Code cloned and identity configured


In [3]:
!pip install -q torch torchvision scipy matplotlib
sys.path.insert(0, "/kaggle/working/ThermoRG-NN/src")
sys.path.insert(0, "/kaggle/working/ThermoRG-NN")
print("Environment ready")

Environment ready


## ▶️ Step 2: Smoke Test

No GPU, no real data — catches import errors, d_manifold computation, JSON/CSV writing. Burns 0 GPU quota.

In [4]:
!python3 experiments/phase_a/smoke_test.py

TEST 1: Imports
  torch/numpy: OK
  torch.utils.data / torchvision: OK
  matplotlib/scipy: OK

TEST 2: d_manifold computation (fake data)
  Fake data: torch.Size([500, 3072]), labels: torch.Size([500])
  SKIPPED (no TAS package)
  d_separable_95: 9.0
  d_manifold: OK

TEST 3: JSON/CSV result writing
  JSON: OK (/tmp/tmpytqa1zgk/phase_a_results.json)
  CSV: OK (/tmp/tmpytqa1zgk/phase_a_summary.csv)
  JSON round-trip: OK

TEST 4: Plotting (matplotlib)
  Plot saved: /tmp/tmpaeevyksp.png (19652 bytes) - OK

ALL SMOKE TESTS PASSED

Ready to run on Kaggle GPU:
  python3 experiments/phase_a/phase_a_analysis.py \
      --device cuda --n-samples 5000 --n-per-arch 500 \
      --actual-results experiments/phase_a/results/training_results.json \
      --output-dir experiments/phase_a/results/



## ▶️ Step 3: Training with Smart Checkpoint Resume

Trains all 15 architectures (or skips those with existing CSV). try/except/finally ensures partial results are always pushed.

In [5]:
try:
    from experiments.lift_test import train
    print("Starting training with checkpoint resume...")
    train.train_phase_a(
        output_dir="experiments/lift_test/results",
        architectures=None  # train all 15
    )
    print("\n🎉 All architectures trained successfully!")
except Exception as e:
    print(f"\n🚨 Training exception: {e}")
    print("Partial results are safe on disk. Will push below.")
finally:
    print("\n📢 [finally] Pushing all completed results to GitHub...")
    results_dir = "experiments/lift_test/results"
    cmds = [
        f"git add {results_dir}/*/*.csv {results_dir}/*/*.pt 2>/dev/null || true",
        'git commit -m "Data: incremental save - training partial results" || true',
        'git push origin main 2>&1 || git push origin master 2>&1 || true'
    ]
    for cmd in cmds:
        res = subprocess.run(cmd, shell=True, capture_output=True, text=True,
                             cwd="/kaggle/working/ThermoRG-NN")
        if res.returncode != 0 and "nothing to commit" not in res.stdout and "nothing to commit" not in res.stderr:
            print(f"  WARN: {res.stderr[-200:]}")
        else:
            print(f"  OK: {cmd[:65]}")
    print("Push complete.")

2026-03-31 02:46:16.528717: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774925176.719968      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774925176.773279      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774925177.237710      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774925177.237751      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774925177.237754      24 computation_placer.cc:177] computation placer alr

Starting training with checkpoint resume...
Phase A: Training all architectures for 30 epochs
Device: cuda


100%|██████████| 170M/170M [00:01<00:00, 86.0MB/s]



Training ThermoNet-3


ThermoNet-3: 100%|██████████| 30/30 [10:47<00:00, 21.57s/it]


...

💾 已落盘: experiments/lift_test/results/ResNet-18-CIFAR/phase_a_metrics.csv

ResNet-18-CIFAR Results:
  Final Test Accuracy: 92.12%
  Best Test Accuracy: 92.18%
  Grokking Epoch: 2
  Parameters: 11,173,962
  Training Time: 1308.2s

Training VGG-11-CIFAR


VGG-11-CIFAR: 100%|██████████| 30/30 [06:07<00:00, 12.24s/it]


💾 已落盘: experiments/lift_test/results/VGG-11-CIFAR/phase_a_metrics.csv

VGG-11-CIFAR Results:
  Final Test Accuracy: 87.80%
  Best Test Accuracy: 87.80%
  Grokking Epoch: 2
  Parameters: 4,509,450
  Training Time: 376.5s

Training DenseNet-40-CIFAR


DenseNet-40-CIFAR:   0%|          | 0/30 [00:00<?, ?it/s]



🚨 Training exception: running_mean should contain 168 elements not 84
Partial results are safe on disk. Will push below.

📢 [finally] Pushing all completed results to GitHub...
  OK: git add experiments/lift_test/results/*/*.csv experiments/lift_te
  OK: git commit -m "Data: incremental save - training partial results"
  OK: git push origin main 2>&1 || git push origin master 2>&1 || true
Push complete.


## ▶️ Step 4: Load Training Results

Reads best accuracy from completed training CSVs. Safe to re-run — just reads, no side effects.

In [6]:
import glob, json, os, csv

PHASE_A_RESULTS_DIR = "experiments/phase_a/results"
os.makedirs(PHASE_A_RESULTS_DIR, exist_ok=True)

csv_files = sorted(glob.glob("experiments/lift_test/results/*/phase_a_metrics.csv"))
print(f"Found {len(csv_files)} training CSV(s):")

actual_results = {}
for csv_path in csv_files:
    arch = csv_path.split("/")[-2]
    with open(csv_path) as f:
        reader = csv.DictReader(f)
        rows = list(reader)
    # Support: 'acc', 'val_acc', or 'test_acc' column
    acc_col = "val_acc" if "val_acc" in rows[0] else ("test_acc" if "test_acc" in rows[0] else "acc")
    best = max(float(r[acc_col]) for r in rows)
    actual_results[arch] = {"best_acc": float(best)}
    print(f"  {arch}: best_{acc_col}={best:.2f}%")

print(f"\nTotal: {len(actual_results)} architectures with results")

with open(f"{PHASE_A_RESULTS_DIR}/training_results.json", "w") as f:
    json.dump(actual_results, f, indent=2)
print(f"Saved to {PHASE_A_RESULTS_DIR}/training_results.json")

Found 14 training CSV(s):
  ReLUFurnace-3: best_val_acc=72.22%
  ReLUFurnace-5: best_val_acc=78.18%
  ReLUFurnace-7: best_val_acc=80.88%
  ReLUFurnace-9: best_val_acc=76.80%
  ResNet-18-CIFAR: best_val_acc=92.02%
  ThermoBot-3: best_val_acc=71.36%
  ThermoBot-5: best_val_acc=78.54%
  ThermoBot-7: best_val_acc=74.48%
  ThermoBot-9: best_val_acc=78.64%
  ThermoNet-3: best_val_acc=77.22%
  ThermoNet-5: best_val_acc=83.28%
  ThermoNet-7: best_val_acc=82.98%
  ThermoNet-9: best_val_acc=80.44%
  VGG-11-CIFAR: best_val_acc=87.50%

Total: 14 architectures with results
Saved to experiments/phase_a/results/training_results.json


## ▶️ Step 5: TAS Profiling (GPU)

Computes d_manifold + alpha/J_topo metrics for all 15 architectures + generates 3 publication plots. Reads from local training results — safe to re-run with --use-fake if results are missing.

In [7]:
r = subprocess.run(
    [
        "python3",
        "experiments/phase_a/phase_a_analysis.py",
        "--device", "cuda",
        "--n-samples", "5000",
        "--n-per-arch", "500",
        "--actual-results", f"{PHASE_A_RESULTS_DIR}/training_results.json",
        "--output-dir", PHASE_A_RESULTS_DIR,
    ],
    capture_output=True,
    text=True,
)
print("=== STDOUT ===")
print(r.stdout[-4000:] if r.stdout else "(empty)")
print("\n=== STDERR ===")
print(r.stderr[-1000:] if r.stderr else "(empty)")
print(f"\nReturn code: {r.returncode}")

=== STDOUT ===
Device: cuda
  Loaded CIFAR-10 data
  Loaded actual results for 14 architectures

Profiling all 15 architectures (n=500 per arch)...
  [ThermoNet-3]........ params=1.06M alpha=0.000 acc=77.2%
  [ThermoNet-5]........ params=2.13M alpha=0.000 acc=83.3%
  [ThermoNet-7]........ params=2.71M alpha=0.000 acc=83.0%
  [ThermoNet-9]........ params=1.31M alpha=0.000 acc=80.4%
  [ThermoBot-3]........ params=1.01M alpha=0.000 acc=71.4%
  [ThermoBot-5]........ params=1.43M alpha=0.000 acc=78.5%
  [ThermoBot-7]........ params=2.40M alpha=0.000 acc=74.5%
  [ThermoBot-9]........ params=2.81M alpha=0.000 acc=78.6%
  [ReLUFurnace-3]........ params=0.08M alpha=0.000 acc=72.2%
  [ReLUFurnace-5]........ params=0.15M alpha=0.000 acc=78.2%
  [ReLUFurnace-7]........ params=0.22M alpha=0.000 acc=80.9%
  [ReLUFurnace-9]........ params=0.30M alpha=0.000 acc=76.8%
  [ResNet-18-CIFAR]........ params=11.17M alpha=0.000 acc=92.0%
  [VGG-11-CIFAR]........ params=4.51M alpha=0.000 acc=87.5%
  [DenseNet-

## ▶️ Step 6: Push All Results to GitHub

Always runs — even if TAS profiling partially failed. Pushes training CSVs + TAS results + plots.

In [8]:
TRAIN_DIR = "experiments/lift_test/results"
cmds = [
    # Push training CSVs + model weights
    f"git add {TRAIN_DIR}/*/*.csv {TRAIN_DIR}/*/*.pt 2>/dev/null || true",
    # Push TAS profiling results + plots
    f"git add {PHASE_A_RESULTS_DIR}/*.csv {PHASE_A_RESULTS_DIR}/*.json {PHASE_A_RESULTS_DIR}/*.png 2>/dev/null || true",
    'git commit -m "Phase A complete: training + TAS metrics + plots from Kaggle" || true',
    'git push origin main 2>&1 || git push origin master 2>&1 || true'
]

for cmd in cmds:
    res = subprocess.run(cmd, shell=True, capture_output=True, text=True,
                         cwd="/kaggle/working/ThermoRG-NN")
    if res.returncode != 0 and "nothing to commit" not in res.stdout and "nothing to commit" not in res.stderr:
        print(f"WARN [{cmd[:40]}...]: {res.stderr[-200:]}")
    else:
        print(f"OK: {cmd[:70]}")

print("\n🎉 Phase A pipeline complete! Check GitHub for results.")

OK: git add experiments/lift_test/results/*/*.csv experiments/lift_test/re
OK: git add experiments/phase_a/results/*.csv experiments/phase_a/results/
OK: git commit -m "Phase A complete: training + TAS metrics + plots from K
OK: git push origin main 2>&1 || git push origin master 2>&1 || true

🎉 Phase A pipeline complete! Check GitHub for results.
